# Musicm8 — one-click Colab training

Run the big cell below after a reset. It mounts Drive, refreshes GitHub, reuses cached tokens/checkpoints, plays a codec reconstruction sanity check, resumes training, and makes a deterministic memory-test sample.

**How to read the two audio players:** if the codec reconstruction sounds like music but the model sample sounds like noise, the codec/data path is good and the model simply needs more learning. The notebook therefore resumes the proof-of-learning run to 6000 steps with infill disabled and a shorter warmup.

> Colab still has to allocate a GPU. If CUDA is unavailable choose Runtime → Change runtime type → GPU, then run the same cell again.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK: SETUP → VERIFY CODEC → TRAIN/RESUME → MEMORY SAMPLE
# Safe defaults for free Colab / Tesla T4.
# ============================================================

import os
import sys
import json
import shutil
import subprocess
import gc
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run_cmd(cmd, *, cwd=None, log_path=None):
    """Stream subprocess output live, save it to Drive, and show a useful tail on failure."""
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd), flush=True)
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    log_f = log_path.open("w", encoding="utf-8") if log_path else None
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            if log_f:
                log_f.write(line)
                log_f.flush()
            tail.append(line.rstrip())
            tail = tail[-80:]
    finally:
        if log_f:
            log_f.close()

    rc = proc.wait()
    if rc != 0:
        print("\n❌ Command failed with exit code", rc)
        if log_path:
            print("Full log:", log_path)
        if tail:
            print("\n--- last output lines ---")
            print("\n".join(tail[-40:]))
        raise RuntimeError(f"Command failed with exit code {rc}: {' '.join(cmd)}")
    return rc

# ---------- Google Drive ----------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO_DIR = DRIVE_ROOT / "audio"
WORK_DIR = DRIVE_ROOT / "work"
LOG_DIR = WORK_DIR / "logs"
MANIFEST = DRIVE_ROOT / "manifest.jsonl"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Fresh runtime copy of GitHub code ----------
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
REPO_DIR = Path("/content/Musicm8")

if (REPO_DIR / ".git").exists():
    print("Refreshing Musicm8 from GitHub...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# ---------- Dependencies ----------
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
    log_path=LOG_DIR / "pip.log",
)

import torch
import torchaudio
from IPython.display import Audio, display

print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is connected. Choose Runtime > Change runtime type > GPU, "
        "then run THIS SAME CELL again."
    )

print("GPU:", torch.cuda.get_device_name(0))

# ---------- Persistent run settings ----------
CODEC = "encodec24"
CLIP_SECONDS = 8
STRIDE_SECONDS = 8

# This is still an overfit/proof-of-learning run, not a production model.
# 2k steps was enough to make the pipeline run, but often not enough to sound musical.
STEPS = 6000
WARMUP_STEPS = 200
BATCH_SIZE = 1
GRAD_ACCUM = 4
SAVE_EVERY = 50
NUM_WORKERS = 0
LOG_EVERY = 10
INFILL_PROB = 0.0
RUN_NAME = "tiny-overfit"
TRAIN_CONFIG = "configs/v2-colab-tiny.json"

TOKENS_DIR = WORK_DIR / f"tokens-{CODEC}"
INDEX = TOKENS_DIR / "index.jsonl"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
LATEST = RUN_DIR / "latest.pt"
RECON = WORK_DIR / "codec_reconstruction.wav"
SAMPLE = WORK_DIR / "sample_memory.wav"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ WORK_DIR :", WORK_DIR)
print("✅ AUDIO_DIR:", AUDIO_DIR)
print("✅ CODEC    :", CODEC)
print("✅ RUN_NAME :", RUN_NAME)
print("✅ RUN_DIR  :", RUN_DIR)
print("✅ Config   :", TRAIN_CONFIG)
print("✅ Target steps:", STEPS)
print("✅ Save every", SAVE_EVERY, "steps")

# ---------- Audio + manifest ----------
exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus"}
audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
)
print(f"✅ Audio files found: {len(audio_files)}")

if not audio_files:
    raise FileNotFoundError(f"No audio files found in {AUDIO_DIR}")

if not MANIFEST.exists() or MANIFEST.stat().st_size == 0:
    with MANIFEST.open("w", encoding="utf-8") as f:
        for p in audio_files:
            caption = p.stem.replace("_", " " ).replace("-", " " )
            f.write(json.dumps(
                {"audio": str(p), "caption": f"music track, {caption}"},
                ensure_ascii=False
            ) + "\n")
    print("✅ Created manifest:", MANIFEST)
else:
    print("✅ Using existing manifest:", MANIFEST)

# ---------- Tokenize only if cache is missing ----------
if INDEX.exists() and INDEX.stat().st_size > 0:
    print("✅ Token cache already exists:", INDEX)
else:
    shutil.rmtree(TOKENS_DIR, ignore_errors=True)
    tokenize_cmd = [
        sys.executable, "-u", "tokenize_dataset.py",
        "--manifest", MANIFEST,
        "--out", TOKENS_DIR,
        "--codec", CODEC,
        "--channels", "1",
        "--clip-seconds", str(CLIP_SECONDS),
        "--stride-seconds", str(STRIDE_SECONDS),
        "--keep-tail",
        "--device", "cuda",
    ]
    print("\n🎵 Tokenizing audio...")
    run_cmd(tokenize_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "tokenize.log")

if not INDEX.exists() or INDEX.stat().st_size == 0:
    raise RuntimeError(f"No usable token index was created at {INDEX}")

print("✅ Tokenization ready:", INDEX)

# ---------- Codec sanity check ----------
# This bypasses the learned model completely. If this player sounds like recognizable
# training music, the audio/token/codec pipeline is healthy and any noise is the model.
if not RECON.exists():
    print("\n🔎 Building codec reconstruction sanity sample...")
    from codec import codec_from_info

    meta = json.loads((TOKENS_DIR / "meta.json").read_text(encoding="utf-8"))
    first_row = json.loads(next(
        line for line in INDEX.read_text(encoding="utf-8").splitlines() if line.strip()
    ))
    payload = torch.load(TOKENS_DIR / first_row["tokens"], map_location="cpu", weights_only=True)
    real_codes = payload["codes"] if isinstance(payload, dict) else payload

    codec = codec_from_info(meta["codec"], torch.device("cuda"))
    recon_wav = codec.decode(real_codes)
    torchaudio.save(RECON, recon_wav, codec.info.sample_rate)
    del codec, recon_wav, real_codes, payload
    gc.collect()
    torch.cuda.empty_cache()

print("\n▶️ CODEC RECONSTRUCTION — this should sound like real/compressed music:")
display(Audio(str(RECON)))

# ---------- Train / resume ----------
train_cmd = [
    sys.executable, "-u", "train.py",
    "--data", INDEX,
    "--config", TRAIN_CONFIG,
    "--out", RUN_DIR,
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--steps", str(STEPS),
    "--warmup-steps", str(WARMUP_STEPS),
    "--save-every", str(SAVE_EVERY),
    "--log-every", str(LOG_EVERY),
    "--num-workers", str(NUM_WORKERS),
    "--infill-prob", str(INFILL_PROB),
    "--device", "cuda",
]

if LATEST.exists():
    try:
        prior = torch.load(LATEST, map_location="cpu", weights_only=False)
        prior_step = int(prior.get("step", 0))
    except Exception:
        prior_step = 0
    print(f"\n♻️ Found checkpoint at step {prior_step}; target is {STEPS}.")
    train_cmd += ["--resume", LATEST]
else:
    prior_step = 0
    print("\n🚀 Starting a fresh training run")

if prior_step < STEPS:
    try:
        run_cmd(train_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "train.log")
    except RuntimeError:
        print("\n⚠️ First training attempt failed. Cleaning GPU memory and retrying once...")
        gc.collect()
        torch.cuda.empty_cache()
        run_cmd(train_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "train_retry.log")
else:
    print("✅ Checkpoint already reached target steps; skipping training.")

if not LATEST.exists():
    raise RuntimeError(f"Training ended but checkpoint was not found at {LATEST}")

print("✅ Checkpoint:", LATEST)

# ---------- Deterministic memorization check ----------
# Use a caption the model actually saw during training and nearly-greedy sampling.
first_manifest_row = json.loads(next(
    line for line in MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()
))
PROMPT = first_manifest_row.get("caption", "music track")
print("\n🧠 Memory-test prompt:", PROMPT)

generate_cmd = [
    sys.executable, "-u", "generate.py",
    "--checkpoint", LATEST,
    "--prompt", PROMPT,
    "--seconds", "4",
    "--temperature", "1.0",
    "--top-k", "1",
    "--top-p", "1.0",
    "--cfg-scale", "1.0",
    "--seed", "42",
    "--out", SAMPLE,
    "--device", "cuda",
]

print("\n🎧 Generating deterministic memory-check sample...")
run_cmd(generate_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "generate_memory.log")

print("\n▶️ MODEL MEMORY SAMPLE — this is the one that should improve after more training:")
display(Audio(str(SAMPLE)))

print("\n✅ MUSICM8 COMPLETE")
print("Codec check:", RECON)
print("Checkpoint :", LATEST)
print("Model sample:", SAMPLE)
print("Logs       :", LOG_DIR)


## Optional: quick status check

Run this only to inspect what is already saved in Drive.


In [ ]:
from pathlib import Path
root = Path('/content/drive/MyDrive/Musicm8')
work = root / 'work'
print('Token indexes:', list(work.glob('tokens-*/index.jsonl')) if work.exists() else [])
print('Checkpoints  :', list(work.glob('runs/*/latest.pt')) if work.exists() else [])
print('WAVs         :', list(work.glob('*.wav')) if work.exists() else [])
print('Logs         :', list((work / 'logs').glob('*.log')) if (work / 'logs').exists() else [])
